# Stage 13: Adaptive Volatility Targeting

This notebook compares a base risky strategy with an adaptive volatility-targeted overlay that allocates unused capital to a defensive sleeve.

## 1. Load Data

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go

project_root = Path.cwd().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.backtesting import RollingBacktester, VolatilityTargetingConfig, apply_volatility_targeting
from src.data_pipeline import DataPreprocessor, YahooFinanceProvider, get_defensive_asset_returns
from src.optimization import HERCAllocator
from src.analytics import PerformanceAnalytics, RiskAnalytics

In [ ]:
symbols = [
    "HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS", "TCS.NS", "INFY.NS",
    "RELIANCE.NS", "ITC.NS", "SUNPHARMA.NS", "LT.NS", "GOLDBEES.NS"
]
start_date = "2018-01-01"
end_date = "2025-01-01"

provider = YahooFinanceProvider()
market_data = provider.get_market_data(symbols, start_date, end_date)
prices_df, quality_summary = DataPreprocessor.handle_missing_values(market_data.prices_df)
returns_df = DataPreprocessor.build_returns_risk_outputs(prices_df).returns_df
returns_df.tail()

## 2. Run Base Strategy Backtest

In [ ]:
allocator = HERCAllocator(covariance_method="ledoit_wolf")
backtester = RollingBacktester(
    allocator=allocator,
    train_window=252,
    rebalance_frequency="M",
)
base_results = backtester.run(returns_df)
base_returns = base_results["portfolio_returns"]
base_returns.tail()

## 3. Fetch Defensive Asset Returns

In [ ]:
defensive_returns, defensive_metadata = get_defensive_asset_returns(
    start_date=returns_df.index.min(),
    end_date=returns_df.index.max(),
    preferred_ticker="LIQUIDBEES.NS",
    fallback_tickers=["LIQUIDETF.NS"],
    synthetic_annual_rate=0.04,
)
defensive_metadata

## 4. Apply Adaptive Volatility Targeting

In [ ]:
config = VolatilityTargetingConfig(
    realized_vol_window=63,
    regime_lookback_window=252,
    base_target_vol=0.10,
    exposure_floor=0.25,
    exposure_cap=1.0,
    no_trade_band=0.05,
)
overlay_results = apply_volatility_targeting(base_returns, defensive_returns, config)
overlay_results["summary"]

## 5. Compare Growth Curves

In [ ]:
base_growth = (1.0 + overlay_results["risky_returns"]).cumprod()
targeted_growth = (1.0 + overlay_results["targeted_returns"]).cumprod()

growth_fig = go.Figure()
growth_fig.add_scatter(x=base_growth.index, y=base_growth.values, mode="lines", name="Base HERC")
growth_fig.add_scatter(x=targeted_growth.index, y=targeted_growth.values, mode="lines", name="HERC + Adaptive VT")
growth_fig.update_layout(title="Base vs Adaptive Vol-Targeted Growth", template="plotly_white")
growth_fig.show()

## 6. Compare Drawdowns

In [ ]:
drawdown_df = pd.DataFrame(
    {
        "Base HERC": RiskAnalytics.drawdown_series(overlay_results["risky_returns"]),
        "HERC + Adaptive VT": RiskAnalytics.drawdown_series(overlay_results["targeted_returns"]),
    }
)
drawdown_df.plot(title="Drawdown Comparison", figsize=(12, 4));

## 7. Compare Metrics

In [ ]:
metrics_df = pd.DataFrame(
    {
        "Base HERC": PerformanceAnalytics.summary_table(overlay_results["risky_returns"]),
        "HERC + Adaptive VT": PerformanceAnalytics.summary_table(overlay_results["targeted_returns"]),
    }
)
metrics_df.loc["final_growth"] = [base_growth.iloc[-1], targeted_growth.iloc[-1]]
metrics_df

## 8. Plot Exposure

In [ ]:
overlay_results["exposure_series"].plot(title="Risky Exposure", figsize=(12, 4));

## 9. Plot Realized vs Target Volatility

In [ ]:
vol_df = pd.DataFrame(
    {
        "Realized Vol": overlay_results["realized_volatility"],
        "Target Vol": overlay_results["target_volatility"],
    }
)
vol_df.plot(title="Realized vs Target Volatility", figsize=(12, 4));

## 10. Interpret Results

Use the tables and plots above to answer:

1. Does adaptive volatility targeting reduce drawdown?
2. Does Calmar improve?
3. How much CAGR is sacrificed?
4. How often is the system calm, normal, stress, or crisis?
5. How much capital moves into the defensive sleeve?
6. Does HERC + adaptive vol targeting outperform plain HERC on drawdown?